In [ ]:
pip install catboost matplotlib scikit-learn pandas

In [ ]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive
import numpy as np
from itertools import product

In [ ]:
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/機器學習小組/new_travel.xlsx'

# 過去6個月

In [ ]:
# 讀取與清洗資料
df = pd.read_excel(file_path)
df = df.dropna()
df.tail(10)

In [ ]:
# 步驟 1：擷取指定月份欄位
months = ['2024-10-31', '2024-11-30', '2024-12-31', '2025-01-31', '2025-02-28', '2025-03-31', '2025-04-30']
metrics = ['subscribers', 'views']
columns_to_select = [f"{month}_{metric}" for month in months for metric in metrics]
df = df.set_index('creator_handle')[columns_to_select]
df

In [ ]:

# 步驟 2：標準化 subscribers/views 數據
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
df_scaled

In [ ]:

# 步驟 3：建立滑動視窗資料集 (前2個月作為歷史 + 後1個月為預測目標)
def generate_windowed_data(df_scaled, months):
  data = []
  for i in range(len(months) - 2):
    m1, m2, m3 = months[i], months[i+1], months[i+2]
    tmp = pd.DataFrame({
      '1st_Subscribers': df_scaled[f"{m1}_subscribers"],
      '1st_Views': df_scaled[f"{m1}_views"],
      '2nd_Subscribers': df_scaled[f"{m2}_subscribers"],
      '2nd_Views': df_scaled[f"{m2}_views"],
      'Views': df_scaled[f"{m3}_views"],
      'TargetMonth': m3
    }, index=df_scaled.index)
    data.append(tmp)
  return pd.concat(data)

dataset = generate_windowed_data(df_scaled, months)
print(dataset.shape)
display(dataset)

In [ ]:
# 步驟 4：分出訓練集、預測 2025-04-30 的測試集

pd.set_option('display.max_columns', None)     # 顯示所有欄位
pd.set_option('display.expand_frame_repr', False)  # 避免欄位換行顯示
pd.set_option('display.max_colwidth', None)    # 避免欄位內容被截斷
pd.set_option('display.width', 180)            # 顯示欄寬總寬度（可依螢幕調整）

train_data = dataset[dataset['TargetMonth'] != '2025-04-30']
predict_april_data = dataset[dataset['TargetMonth'] == '2025-04-30']

X = train_data[['1st_Views', '1st_Subscribers', '2nd_Views', '2nd_Subscribers']]
print("X.shape:",X.shape)
print("train_data:")
display(X.head(10))
y = train_data['Views']
print("train_data_label:")
display(y.head(10))

X_april = predict_april_data[['1st_Views', '1st_Subscribers', '2nd_Views', '2nd_Subscribers']]
print("test_data:")
display(X_april.head(10))
y_april_true = predict_april_data['Views']
print("test_data_label:")
display(y_april_true.head(10))

In [ ]:

# 步驟 5：分割訓練與測試集 (70/30)，並訓練CatBoost模型，並找出最佳參數組合。

## 切分資料（70% train, 30% test）
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 參數組合清單
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.06, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [100, 150, 200]
}

best_score = -np.inf
best_params = None
best_model = None

## 使用for迴圈，尋找最佳參數組合，將最佳參數組合存入best model。
for depth, lr, reg, iters in product(param_grid['depth'],
                      param_grid['learning_rate'],
                      param_grid['l2_leaf_reg'],
                      param_grid['iterations']):
    model = CatBoostRegressor(
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=reg,
        iterations=iters,
        verbose=0,
        random_state=42
    )
    model.fit(X_train_final, y_train_final)
    y_test_pred = model.predict(X_test_final)
    r2 = r2_score(y_test_final, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test_final, y_test_pred))

    if r2 > best_score:
        best_score = r2
        best_params = {
            'depth': depth,
            'learning_rate': lr,
            'l2_leaf_reg': reg,
            'iterations': iters
        }
        best_model = model
        best_rmse_test = rmse_test

print("Best model")
print("最佳參數組合：", best_params)
print(f"Test R²： {best_score:.4f}")
print(f"Test RMSE: {best_rmse_test:.4f}")


In [ ]:
# 步驟 6: 使用best model預測 4 月資料，並計算RMSE、R-square score。

y_april_pred = best_model.predict(X_april)
rmse_april = np.sqrt(mean_squared_error(y_april_true, y_april_pred))
r2_april = r2_score(y_april_true, y_april_pred)
print("成效評估 ：計算April的實際值與預測值的 RMSE 與 R-square score 。")
print(f"April RMSE: {rmse_april:.4f}")
print(f"April R² Score: {r2_april:.4f}")

命中率 nDCG

In [ ]:

# 將實際值與預測值轉為帶 creator index 的 Series
creator_april = predict_april_data.index
y_april_true_series = pd.Series(y_april_true, index=creator_april)
y_april_pred_series = pd.Series(y_april_pred, index=creator_april)

# Top-k 設定為前 10%
top_k = int(len(y_april_true_series) * 0.10)

# 取出真實與預測的前 top_k 名創作者 index
actual_top_indices = y_april_true_series.sort_values(ascending=False).head(top_k).index
predicted_top_indices = y_april_pred_series.sort_values(ascending=False).head(top_k).index

# 命中率計算
hit_count = len(set(actual_top_indices) & set(predicted_top_indices))
hit_rate = hit_count / top_k

# 定義 DCG 計算函數（使用真實觀看數作為 gain）
def dcg_score(y_true_series, y_pred_series, k):
    order = y_pred_series.sort_values(ascending=False).index[:k]
    y_true_at_k = y_true_series.loc[order].values
    gains = np.log1p(y_true_at_k)  # log(1 + gain) 防止大值主導
    discounts = np.log2(np.arange(2, k + 2))  # log2 折扣
    return np.sum(gains / discounts)

# 計算 DCG 與 ideal DCG，得到 nDCG
dcg = dcg_score(y_april_true_series, y_april_pred_series, top_k)
ideal_dcg = dcg_score(y_april_true_series, y_april_true_series, top_k)
ndcg = dcg / ideal_dcg if ideal_dcg != 0 else 0

# 輸出評估指標
print("\n📊 April 排名指標評估（前 10%）")
print(f"命中率 (HitRate): {hit_rate:.3f}")
print(f"標準化折扣累積增益 (nDCG): {ndcg:.3f}")


##命中率、前10%是否良好指標分析

In [ ]:
# 把預測值加進 predict_april_data DataFrame 中
predict_april_data = predict_april_data.copy()
predict_april_data["PredictedViews"] = y_april_pred

# 設定前 20% 門檻
top_k_percent = 0.2
top_n = int(len(predict_april_data) * top_k_percent)

# 取出預測值最高的前 20% 創作者
top_creators = predict_april_data.sort_values("PredictedViews", ascending=False).head(top_n)

# 也抓出預測值最低的後 20% 作為對照組
bottom_creators = predict_april_data.sort_values("PredictedViews", ascending=True).head(top_n)

# 計算實際觀看數（標準化值）的平均
top_avg = top_creators["Views"].mean()
bottom_avg = bottom_creators["Views"].mean()
overall_avg = predict_april_data["Views"].mean()

# 輸出分析結果
print("🔥 預測前 20% 創作者的平均實際觀看數（標準化）:", round(top_avg, 3))
print("📉 預測後 20% 創作者的平均實際觀看數:", round(bottom_avg, 3))
print("⚠️ 全體平均實際觀看數:", round(overall_avg, 3))

# 顯示前幾名潛力創作者的預測與實際值
print("\n📋 前幾名潛力創作者（標準化值）:")
display(top_creators[["PredictedViews", "Views"]].head(10))


# 過去9個月

In [ ]:
# 讀取與清洗資料
df = pd.read_excel(file_path)
df = df.dropna()
df.tail(10)

In [ ]:
# 步驟 1：擷取指定月份欄位
months = ['2024-07-31', '2024-08-31', '2024-09-30', '2024-10-31', '2024-11-30', '2024-12-31', '2025-01-31', '2025-02-28', '2025-03-31', '2025-04-30']
metrics = ['subscribers', 'views']
columns_to_select = [f"{month}_{metric}" for month in months for metric in metrics]
df = df.set_index('creator_handle')[columns_to_select]
df

In [ ]:
# 步驟 2：標準化 subscribers/views 數據
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
df_scaled

In [ ]:
# 步驟 3：建立滑動視窗資料集 (2個月作為歷史 + 1個月為預測目標)
def generate_windowed_data(df_scaled, months):
  data = []
  for i in range(len(months) - 2):
    m1, m2, m3 = months[i], months[i+1], months[i+2]
    tmp = pd.DataFrame({
      '1st_Subscribers': df_scaled[f"{m1}_subscribers"],
      '1st_Views': df_scaled[f"{m1}_views"],
      '2nd_Subscribers': df_scaled[f"{m2}_subscribers"],
      '2nd_Views': df_scaled[f"{m2}_views"],
      'Views': df_scaled[f"{m3}_views"],
      'TargetMonth': m3
    }, index=df_scaled.index)
    data.append(tmp)
  return pd.concat(data)

dataset = generate_windowed_data(df_scaled, months)
print(dataset.shape)
display(dataset)

In [ ]:
# 步驟 4：分出訓練集、預測 2025-04-30 的測試集

pd.set_option('display.max_columns', None)     # 顯示所有欄位
pd.set_option('display.expand_frame_repr', False)  # 避免欄位換行顯示
pd.set_option('display.max_colwidth', None)    # 避免欄位內容被截斷
pd.set_option('display.width', 180)            # 顯示欄寬總寬度（可依螢幕調整）

train_data = dataset[dataset['TargetMonth'] != '2025-04-30']
predict_april_data = dataset[dataset['TargetMonth'] == '2025-04-30']

X = train_data[['1st_Views', '1st_Subscribers', '2nd_Views', '2nd_Subscribers']]
print("X.shape:",X.shape)
print("train_data:")
display(X.head(10))
y = train_data['Views']
print("train_data_label:")
display(y.head(10))

X_april = predict_april_data[['1st_Views', '1st_Subscribers', '2nd_Views', '2nd_Subscribers']]
print("test_data:")
display(X_april.head(10))
y_april_true = predict_april_data['Views']
print("test_data_label:")
display(y_april_true.head(10))

In [ ]:
# 步驟 5：分割訓練與測試集 (70/30)，並訓練CatBoost模型，並找出最佳參數組合。

## 切分資料（70% train, 30% test）
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 參數組合清單
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.06, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [100, 150, 200]
}

best_score = -np.inf
best_params = None
best_model = None

## 使用for迴圈，尋找最佳參數組合，將最佳參數組合存入best model。
for depth, lr, reg, iters in product(param_grid['depth'],
                      param_grid['learning_rate'],
                      param_grid['l2_leaf_reg'],
                      param_grid['iterations']):
    model = CatBoostRegressor(
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=reg,
        iterations=iters,
        verbose=0,
        random_state=42
    )
    model.fit(X_train_final, y_train_final)
    y_test_pred = model.predict(X_test_final)
    r2 = r2_score(y_test_final, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test_final, y_test_pred))

    if r2 > best_score:
        best_score = r2
        best_params = {
            'depth': depth,
            'learning_rate': lr,
            'l2_leaf_reg': reg,
            'iterations': iters
        }
        best_model = model
        best_rmse_test = rmse_test

print("Best model")
print("最佳參數組合：", best_params)
print(f"Test R²： {best_score:.4f}")
print(f"Test RMSE: {best_rmse_test:.4f}")


In [ ]:
# 步驟 6: 使用best model預測 4 月資料，並計算RMSE、R-square score。

y_april_pred = best_model.predict(X_april)
rmse_april = np.sqrt(mean_squared_error(y_april_true, y_april_pred))
r2_april = r2_score(y_april_true, y_april_pred)
print("成效評估 ：計算April的實際值與預測值的 RMSE 與 R-square score 。")
print(f"April RMSE: {rmse_april:.4f}")
print(f"April R² Score: {r2_april:.4f}")

In [ ]:
# 測試集圖
plt.subplot(1, 2, 1)
plt.scatter(y_test_final, y_test_pred, alpha=0.7)
plt.plot([y_test_final.min(), y_test_final.max()], [y_test_final.min(), y_test_final.max()], 'r--')
plt.title("Test Pedict vs Actual")
plt.xlabel("Actual value")
plt.ylabel("Predict value")

# 四月預測圖
plt.subplot(1, 2, 2)
plt.scatter(y_april_true, y_april_pred, alpha=0.7)
plt.plot([y_april_true.min(), y_april_true.max()], [y_april_true.min(), y_april_true.max()], 'r--')
plt.title("2025-04-30 Predict vs Actual")
plt.xlabel("Actual value")
plt.ylabel("Predict value")

plt.tight_layout()
plt.show()

命中率和nDCG

In [ ]:

# 將實際值與預測值轉為帶 creator index 的 Series
creator_april = predict_april_data.index
y_april_true_series = pd.Series(y_april_true, index=creator_april)
y_april_pred_series = pd.Series(y_april_pred, index=creator_april)

# Top-k 設定為前 10%
top_k = int(len(y_april_true_series) * 0.10)

# 取出真實與預測的前 top_k 名創作者 index
actual_top_indices = y_april_true_series.sort_values(ascending=False).head(top_k).index
predicted_top_indices = y_april_pred_series.sort_values(ascending=False).head(top_k).index

# 命中率計算
hit_count = len(set(actual_top_indices) & set(predicted_top_indices))
hit_rate = hit_count / top_k

# 定義 DCG 計算函數（使用真實觀看數作為 gain）
def dcg_score(y_true_series, y_pred_series, k):
    order = y_pred_series.sort_values(ascending=False).index[:k]
    y_true_at_k = y_true_series.loc[order].values
    gains = np.log1p(y_true_at_k)  # log(1 + gain) 防止大值主導
    discounts = np.log2(np.arange(2, k + 2))  # log2 折扣
    return np.sum(gains / discounts)

# 計算 DCG 與 ideal DCG，得到 nDCG
dcg = dcg_score(y_april_true_series, y_april_pred_series, top_k)
ideal_dcg = dcg_score(y_april_true_series, y_april_true_series, top_k)
ndcg = dcg / ideal_dcg if ideal_dcg != 0 else 0

# 輸出評估指標
print("\n📊 April 排名指標評估（前 10%）")
print(f"命中率 (HitRate): {hit_rate:.3f}")
print(f"標準化折扣累積增益 (nDCG): {ndcg:.3f}")


##命中率、前10%是否良好指標分析

In [ ]:
# 潛力創作者名單與實際驗證 ===

# 把預測值加進 predict_april_data DataFrame 中
predict_april_data = predict_april_data.copy()
predict_april_data["PredictedViews"] = y_april_pred

# 設定前 20% 門檻
top_k_percent = 0.2
top_n = int(len(predict_april_data) * top_k_percent)

# 取出預測值最高的前 20% 創作者
top_creators = predict_april_data.sort_values("PredictedViews", ascending=False).head(top_n)

# 也抓出預測值最低的後 20% 作為對照組
bottom_creators = predict_april_data.sort_values("PredictedViews", ascending=True).head(top_n)

# 計算實際觀看數（標準化值）的平均
top_avg = top_creators["Views"].mean()
bottom_avg = bottom_creators["Views"].mean()
overall_avg = predict_april_data["Views"].mean()

# 輸出分析結果
print("🔥 預測前 20% 創作者的平均實際觀看數（標準化）:", round(top_avg, 3))
print("📉 預測後 20% 創作者的平均實際觀看數:", round(bottom_avg, 3))
print("⚠️ 全體平均實際觀看數:", round(overall_avg, 3))

# 顯示前幾名潛力創作者的預測與實際值
print("\n📋 前幾名潛力創作者（標準化值）:")
display(top_creators[["PredictedViews", "Views"]].head(10))


In [ ]:
# === 將標準化的預測值與實際值還原（反標準化） ===
def inverse_transform(series, column_name):
    idx = df.columns.get_loc(column_name)
    return scaler.mean_[idx] + scaler.scale_[idx] * series

# 抓出對應的實際值與預測值（反標準化）
actual_views_april = inverse_transform(y_april_true, '2025-04-30_views')
predicted_views_april = inverse_transform(y_april_pred, '2025-04-30_views')

# 組成表格
results_df = pd.DataFrame({
    "creator_handle": predict_april_data.index,
    "Actual_Views_2025_04": actual_views_april,
    "Predicted_Views_2025_04": predicted_views_april
})

# 顯示前幾列確認正確
display(results_df.head())

# 儲存成 CSV 檔案
save_path = '/content/drive/MyDrive/機器學習小組/views_predict.csv'

# 儲存為 CSV
results_df.to_csv(save_path, index=False)

print(f"✅ 檔案已儲存至：{save_path}")


# 過去12個月

In [ ]:
# 讀取與清洗資料
df = pd.read_excel(file_path)
df = df.dropna()
df.tail(10)

In [ ]:
# 步驟 1：擷取指定月份欄位
months = ['2024-04-30', '2024-05-31', '2024-06-30', '2024-07-31', '2024-08-31', '2024-09-30', '2024-10-31', '2024-11-30', '2024-12-31', '2025-01-31', '2025-02-28', '2025-03-31', '2025-04-30']
metrics = ['subscribers', 'views']
columns_to_select = [f"{month}_{metric}" for month in months for metric in metrics]
df = df.set_index('creator_handle')[columns_to_select]
df

In [ ]:
# 步驟 2：標準化 subscribers/views 數據
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
df_scaled

In [ ]:
# 步驟 3：建立滑動視窗資料集 (2個月作為歷史 + 1個月為預測目標)
def generate_windowed_data(df_scaled, months):
  data = []
  for i in range(len(months) - 2):
    m1, m2, m3 = months[i], months[i+1], months[i+2]
    tmp = pd.DataFrame({
      '1st_Subscribers': df_scaled[f"{m1}_subscribers"],
      '1st_Views': df_scaled[f"{m1}_views"],
      '2nd_Subscribers': df_scaled[f"{m2}_subscribers"],
      '2nd_Views': df_scaled[f"{m2}_views"],
      'Views': df_scaled[f"{m3}_views"],
      'TargetMonth': m3
    }, index=df_scaled.index)
    data.append(tmp)
  return pd.concat(data)

dataset = generate_windowed_data(df_scaled, months)
print(dataset.shape)
display(dataset)

In [ ]:
# 步驟 4：分出訓練集、預測 2025-04-30 的測試集

pd.set_option('display.max_columns', None)     # 顯示所有欄位
pd.set_option('display.expand_frame_repr', False)  # 避免欄位換行顯示
pd.set_option('display.max_colwidth', None)    # 避免欄位內容被截斷
pd.set_option('display.width', 180)            # 顯示欄寬總寬度（可依螢幕調整）

train_data = dataset[dataset['TargetMonth'] != '2025-04-30']
predict_april_data = dataset[dataset['TargetMonth'] == '2025-04-30']

X = train_data[['1st_Views', '1st_Subscribers', '2nd_Views', '2nd_Subscribers']]
print("X.shape:",X.shape)
print("train_data:")
display(X.head(10))
y = train_data['Views']
print("train_data_label:")
display(y.head(10))

X_april = predict_april_data[['1st_Views', '1st_Subscribers', '2nd_Views', '2nd_Subscribers']]
print("test_data:")
display(X_april.head(10))
y_april_true = predict_april_data['Views']
print("test_data_label:")
display(y_april_true.head(10))

In [ ]:
# 步驟 5：分割訓練與測試集 (70/30)，並訓練CatBoost模型，並找出最佳參數組合。

## 切分資料（70% train, 30% test）
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 參數組合清單
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.06, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [100, 150, 200]
}

best_score = -np.inf
best_params = None
best_model = None

## 使用for迴圈，尋找最佳參數組合，將最佳參數組合存入best model。
for depth, lr, reg, iters in product(param_grid['depth'],
                      param_grid['learning_rate'],
                      param_grid['l2_leaf_reg'],
                      param_grid['iterations']):
    model = CatBoostRegressor(
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=reg,
        iterations=iters,
        verbose=0,
        random_state=42
    )
    model.fit(X_train_final, y_train_final)
    y_test_pred = model.predict(X_test_final)
    r2 = r2_score(y_test_final, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test_final, y_test_pred))

    if r2 > best_score:
        best_score = r2
        best_params = {
            'depth': depth,
            'learning_rate': lr,
            'l2_leaf_reg': reg,
            'iterations': iters
        }
        best_model = model
        best_rmse_test = rmse_test

print("Best model")
print("最佳參數組合：", best_params)
print(f"Test R²： {best_score:.4f}")
print(f"Test RMSE: {best_rmse_test:.4f}")


In [ ]:
# 步驟 6: 使用best model預測 4 月資料，並計算RMSE、R-square score。

y_april_pred = best_model.predict(X_april)
rmse_april = np.sqrt(mean_squared_error(y_april_true, y_april_pred))
r2_april = r2_score(y_april_true, y_april_pred)
print("成效評估 ：計算April的實際值與預測值的 RMSE 與 R-square score 。")
print(f"April RMSE: {rmse_april:.4f}")
print(f"April R² Score: {r2_april:.4f}")

命中率和nDCG

In [ ]:

# 將實際值與預測值轉為帶 creator index 的 Series
creator_april = predict_april_data.index
y_april_true_series = pd.Series(y_april_true, index=creator_april)
y_april_pred_series = pd.Series(y_april_pred, index=creator_april)

# Top-k 設定為前 10%
top_k = int(len(y_april_true_series) * 0.10)

# 取出真實與預測的前 top_k 名創作者 index
actual_top_indices = y_april_true_series.sort_values(ascending=False).head(top_k).index
predicted_top_indices = y_april_pred_series.sort_values(ascending=False).head(top_k).index

# 命中率計算
hit_count = len(set(actual_top_indices) & set(predicted_top_indices))
hit_rate = hit_count / top_k

# 定義 DCG 計算函數（使用真實觀看數作為 gain）
def dcg_score(y_true_series, y_pred_series, k):
    order = y_pred_series.sort_values(ascending=False).index[:k]
    y_true_at_k = y_true_series.loc[order].values
    gains = np.log1p(y_true_at_k)  # log(1 + gain) 防止大值主導
    discounts = np.log2(np.arange(2, k + 2))  # log2 折扣
    return np.sum(gains / discounts)

# 計算 DCG 與 ideal DCG，得到 nDCG
dcg = dcg_score(y_april_true_series, y_april_pred_series, top_k)
ideal_dcg = dcg_score(y_april_true_series, y_april_true_series, top_k)
ndcg = dcg / ideal_dcg if ideal_dcg != 0 else 0

# 輸出評估指標
print("\n📊 April 排名指標評估（前 10%）")
print(f"命中率 (HitRate): {hit_rate:.3f}")
print(f"標準化折扣累積增益 (nDCG): {ndcg:.3f}")

## 12個月綜合指標分數

In [ ]:
file_sub = '/content/drive/MyDrive/機器學習小組/april_cat12subscribers_predictions.xlsx'
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
df = y_april_pred_series.reset_index()  # 把 creator_handle 從 index 變成欄位
df.columns = ['creator_handle', 'view_score']  # 重命名欄位

df1 = pd.read_excel(file_path)
scaler = StandardScaler()

# 對欄位 '2025-04-30 videos' 做標準化
df1['2025-04-30_videos'] = scaler.fit_transform(df1[['2025-04-30_videos']])
df2 = pd.read_excel(file_sub)

df['綜合指標'] = 0.1 * df1['2025-04-30_videos'] + 0.4 * df['view_score'] + 0.5 * df2['score']
# 按照「綜合指標」由高到低排序
df_sorted = df.sort_values(by='綜合指標', ascending=False)

print(df_sorted.head(20))




##命中率、前10%是否良好指標分析

In [ ]:
#潛力創作者名單與實際驗證

# 把預測值加進 predict_april_data DataFrame 中
predict_april_data = predict_april_data.copy()
predict_april_data["PredictedViews"] = y_april_pred

# 設定前 20% 門檻
top_k_percent = 0.2
top_n = int(len(predict_april_data) * top_k_percent)

# 取出預測值最高的前 20% 創作者
top_creators = predict_april_data.sort_values("PredictedViews", ascending=False).head(top_n)

# 也抓出預測值最低的後 20% 作為對照組
bottom_creators = predict_april_data.sort_values("PredictedViews", ascending=True).head(top_n)

# 計算實際觀看數（標準化值）的平均
top_avg = top_creators["Views"].mean()
bottom_avg = bottom_creators["Views"].mean()
overall_avg = predict_april_data["Views"].mean()

# 輸出分析結果
print("🔥 預測前 20% 創作者的平均實際觀看數（標準化）:", round(top_avg, 3))
print("📉 預測後 20% 創作者的平均實際觀看數:", round(bottom_avg, 3))
print("⚠️ 全體平均實際觀看數:", round(overall_avg, 3))

# 顯示前幾名潛力創作者的預測與實際值
print("\n📋 前幾名潛力創作者（標準化值）:")
display(top_creators[["PredictedViews", "Views"]].head(10))
